In [1]:
!pip install google-adk

In [2]:
!pip install -U typing_extensions pydantic pydantic-core google-genai google-adk


In [3]:
!python -m pip install --upgrade --force-reinstall "typing_extensions>=4.9"



  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.25 requires openai<2.0.0,>=1.86.0, but you have openai 2.8.1 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [4]:
!python -m pip install --upgrade google-adk google-genai


In [5]:
!pip install nest_asyncio


In [ ]:
import os
import asyncio
from typing import Optional, Dict, Any

from google.adk.agents import Agent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.tool_context import ToolContext
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types

os.environ["GOOGLE_API_KEY"] = "na"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"

APP_NAME = "weather_tutorial_app"
USER_ID_STATEFUL = "user_state_demo"
SESSION_ID_STATEFUL = "session_state_demo_001"

def say_hello(name: Optional[str] = None) -> str:
    if name:
        greeting = f"Hello, {name}!"
        print(f"--- Tool: say_hello called with name: {name} ---")
    else:
        greeting = "Hello there!"
        print(f"--- Tool: say_hello called without a specific name ---")
    return greeting

def say_goodbye() -> str:
    print(f"--- Tool: say_goodbye called ---")
    return "Goodbye! Have a great day."

def get_weather_stateful(city: str, tool_context: ToolContext) -> dict:
    print(f"--- Tool: get_weather_stateful called for {city} ---")
    
    preferred_unit = tool_context.state.get("user_preference_temperature_unit", "Celsius")
    print(f"--- Tool: Reading state 'user_preference_temperature_unit': {preferred_unit} ---")
    
    city_normalized = city.lower().replace(" ", "")
    
    mock_weather_db = {
        "newyork": {"temp_c": 25, "condition": "sunny"},
        "london": {"temp_c": 15, "condition": "cloudy"},
        "tokyo": {"temp_c": 18, "condition": "light rain"},
    }
    
    if city_normalized in mock_weather_db:
        data = mock_weather_db[city_normalized]
        temp_c = data["temp_c"]
        condition = data["condition"]
        
        if preferred_unit == "Fahrenheit":
            temp_value = (temp_c * 9/5) + 32
            temp_unit = "°F"
        else:
            temp_value = temp_c
            temp_unit = "°C"
        
        report = f"The weather in {city.capitalize()} is {condition} with a temperature of {temp_value:.0f}{temp_unit}."
        result = {"status": "success", "report": report}
        print(f"--- Tool: Generated report in {preferred_unit}. Result: {result} ---")
        
        tool_context.state["last_city_checked_stateful"] = city
        print(f"--- Tool: Updated state 'last_city_checked_stateful': {city} ---")
        
        return result
    else:
        error_msg = f"Sorry, I don't have weather information for '{city}'."
        print(f"--- Tool: City '{city}' not found. ---")
        return {"status": "error", "error_message": error_msg}

def block_keyword_guardrail(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    agent_name = callback_context.agent_name
    print(f"--- Callback: block_keyword_guardrail running for agent: {agent_name} ---")
    
    last_user_message_text = ""
    if llm_request.contents:
        for content in reversed(llm_request.contents):
            if content.role == 'user' and content.parts:
                if content.parts[0].text:
                    last_user_message_text = content.parts[0].text
                    break
    
    print(f"--- Callback: Inspecting last user message: '{last_user_message_text[:100]}...' ---")
    
    keyword_to_block = "BLOCK"
    if keyword_to_block in last_user_message_text.upper():
        print(f"--- Callback: Found '{keyword_to_block}'. Blocking LLM call! ---")
        callback_context.state["guardrail_block_keyword_triggered"] = True
        print(f"--- Callback: Set state 'guardrail_block_keyword_triggered': True ---")
        
        return LlmResponse(
            content=types.Content(
                role="model",
                parts=[types.Part(text=f"I cannot process this request because it contains the blocked keyword '{keyword_to_block}'.")],
            )
        )
    else:
        print(f"--- Callback: Keyword not found. Allowing LLM call for {agent_name}. ---")
        return None

def block_paris_tool_guardrail(tool: BaseTool, args: Dict[str, Any], tool_context: ToolContext) -> Optional[Dict]:
    tool_name = tool.name
    agent_name = tool_context.agent_name
    print(f"--- Callback: block_paris_tool_guardrail running for tool '{tool_name}' in agent '{agent_name}' ---")
    print(f"--- Callback: Inspecting args: {args} ---")
    
    target_tool_name = "get_weather_stateful"
    blocked_city = "paris"
    
    if tool_name == target_tool_name:
        city_argument = args.get("city", "")
        if city_argument and city_argument.lower() == blocked_city:
            print(f"--- Callback: Detected blocked city '{city_argument}'. Blocking tool execution! ---")
            tool_context.state["guardrail_tool_block_triggered"] = True
            print(f"--- Callback: Set state 'guardrail_tool_block_triggered': True ---")
            
            return {
                "status": "error",
                "error_message": f"Policy restriction: Weather checks for '{city_argument.capitalize()}' are currently disabled by a tool guardrail."
            }
        else:
            print(f"--- Callback: City '{city_argument}' is allowed for tool '{tool_name}'. ---")
    else:
        print(f"--- Callback: Tool '{tool_name}' is not the target tool. Allowing. ---")
    
    print(f"--- Callback: Allowing tool '{tool_name}' to proceed. ---")
    return None

greeting_agent = Agent(
    model=MODEL_GEMINI_2_5_FLASH,
    name="greeting_agent",
    instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
    description="Handles simple greetings and hellos using the 'say_hello' tool.",
    tools=[say_hello],
)

farewell_agent = Agent(
    model=MODEL_GEMINI_2_5_FLASH,
    name="farewell_agent",
    instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
    description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
    tools=[say_goodbye],
)

root_agent_tool_guardrail = Agent(
    name="weather_agent_v6_tool_guardrail",
    model=MODEL_GEMINI_2_5_FLASH,
    description="Main agent: Handles weather, delegates, includes input AND tool guardrails.",
    instruction="You are the main Weather Agent. Provide weather using 'get_weather_stateful'. "
                "Delegate greetings to 'greeting_agent' and farewells to 'farewell_agent'. "
                "Handle only weather, greetings, and farewells.",
    tools=[get_weather_stateful],
    sub_agents=[greeting_agent, farewell_agent],
    output_key="last_weather_report",
    before_model_callback=block_keyword_guardrail,
    before_tool_callback=block_paris_tool_guardrail
)

async def setup_session():
    session_service = InMemorySessionService()
    
    initial_state = {
        "user_preference_temperature_unit": "Celsius"
    }
    
    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID_STATEFUL,
        session_id=SESSION_ID_STATEFUL,
        state=initial_state
    )
    print(f"✅ Session created: App='{APP_NAME}', User='{USER_ID_STATEFUL}', Session='{SESSION_ID_STATEFUL}'")
    
    runner = Runner(
        agent=root_agent_tool_guardrail,
        app_name=APP_NAME,
        session_service=session_service
    )
    print(f"✅ Runner created for agent '{runner.agent.name}'")
    
    return session_service, runner

async def call_agent_async(query: str, runner, user_id, session_id):
    print(f"\n>>> User Query: {query}")
    
    content = types.Content(role='user', parts=[types.Part(text=query)])
    final_response_text = "Agent did not produce a final response."
    
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            elif event.actions and event.actions.escalate:
                final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
            break
    
    print(f"<<< Agent Response: {final_response_text}")

async def main():
    session_service, runner = await setup_session()
    
    print("\n" + "="*80)
    print("Testing Tool Argument Guardrail ('Paris' blocked)")
    print("="*80)
    
    print("\n--- Turn 1: Requesting weather in New York (expect allowed) ---")
    await call_agent_async(
        "What's the weather in New York?",
        runner,
        USER_ID_STATEFUL,
        SESSION_ID_STATEFUL
    )
    
    print("\n--- Turn 2: Requesting weather in Paris (expect blocked by tool guardrail) ---")
    await call_agent_async(
        "How about Paris?",
        runner,
        USER_ID_STATEFUL,
        SESSION_ID_STATEFUL
    )
    
    print("\n--- Turn 3: Requesting weather in London (expect allowed) ---")
    await call_agent_async(
        "Tell me the weather in London.",
        runner,
        USER_ID_STATEFUL,
        SESSION_ID_STATEFUL
    )
    
    print("\n" + "="*80)
    print("Inspecting Final Session State")
    print("="*80)
    
    final_session = await session_service.get_session(
        app_name=APP_NAME,
        user_id=USER_ID_STATEFUL,
        session_id=SESSION_ID_STATEFUL
    )
    
    if final_session:
        print(f"Tool Guardrail Triggered: {final_session.state.get('guardrail_tool_block_triggered', 'Not Set')}")
        print(f"Last Weather Report: {final_session.state.get('last_weather_report', 'Not Set')}")
        print(f"Temperature Unit: {final_session.state.get('user_preference_temperature_unit', 'Not Set')}")
    else:
        print("Error: Could not retrieve final session state.")

if __name__ == "__main__":
    asyncio.run(main())

RuntimeError: asyncio.run() cannot be called from a running event loop

In [7]:
await main()


✅ Session created: App='weather_tutorial_app', User='user_state_demo', Session='session_state_demo_001'
✅ Runner created for agent 'weather_agent_v6_tool_guardrail'

Testing Tool Argument Guardrail ('Paris' blocked)

--- Turn 1: Requesting weather in New York (expect allowed) ---

>>> User Query: What's the weather in New York?
--- Callback: block_keyword_guardrail running for agent: weather_agent_v6_tool_guardrail ---
--- Callback: Inspecting last user message: 'What's the weather in New York?...' ---
--- Callback: Keyword not found. Allowing LLM call for weather_agent_v6_tool_guardrail. ---


/Users/ryanxavier/Downloads/simple-nn/.conda/lib/python3.11/site-packages/google/adk/flows/llm_flows/base_llm_flow.py:449: UserWarning: [EXPERIMENTAL] feature FeatureName.PROGRESSIVE_SSE_STREAMING is enabled.
  async for event in agen:


--- Callback: block_paris_tool_guardrail running for tool 'get_weather_stateful' in agent 'weather_agent_v6_tool_guardrail' ---
--- Callback: Inspecting args: {'city': 'New York'} ---
--- Callback: City 'New York' is allowed for tool 'get_weather_stateful'. ---
--- Callback: Allowing tool 'get_weather_stateful' to proceed. ---
--- Tool: get_weather_stateful called for New York ---
--- Tool: Reading state 'user_preference_temperature_unit': Celsius ---
--- Tool: Generated report in Celsius. Result: {'status': 'success', 'report': 'The weather in New york is sunny with a temperature of 25°C.'} ---
--- Tool: Updated state 'last_city_checked_stateful': New York ---
--- Callback: block_keyword_guardrail running for agent: weather_agent_v6_tool_guardrail ---
--- Callback: Inspecting last user message: 'What's the weather in New York?...' ---
--- Callback: Keyword not found. Allowing LLM call for weather_agent_v6_tool_guardrail. ---
<<< Agent Response: The weather in New York is sunny with a 